## Data Cleaning

In [62]:
import pandas as pd

data_path= "../data/raw/csv_files"

# reading all the csv files
customers= pd.read_csv(data_path + "/olist_customers_dataset.csv")
geolocation= pd.read_csv(data_path + "/olist_geolocation_dataset.csv")
order_items= pd.read_csv(data_path + "/olist_order_items_dataset.csv")
order_payments= pd.read_csv(data_path + "/olist_order_payments_dataset.csv")
order_reviews= pd.read_csv(data_path + "/olist_order_reviews_dataset.csv")
orders= pd.read_csv(data_path + "/olist_orders_dataset.csv")
products= pd.read_csv(data_path + "/olist_products_dataset.csv")
sellers= pd.read_csv(data_path + "/olist_sellers_dataset.csv")
category_translation= pd.read_csv(data_path + "/product_category_name_translation.csv")

In [63]:
# making a dictionary of all the files to do certain operations combinely
dfs={"customers":customers,
     "geolocation":geolocation,
     "order_items": order_items,
     "order_payments": order_payments,
     "order_reviews": order_reviews,
     "orders": orders,
     "products": products,
     "sellers": sellers,
     "category_translation": category_translation}

#### Duplicates handling

In [64]:
# removing duplicates from geolocation table
geolocation= geolocation.drop_duplicates()

# to check whether the duplicates are dropped or not, use this code
geolocation.duplicated().sum()

np.int64(0)

In [65]:
#duplicates for this zip_code_prefix column
geolocation['geolocation_zip_code_prefix'].duplicated().sum()

np.int64(719317)

In [66]:
# Making geolocation_zip_code_prefix as unique by aggregating the lat and long columns to avoid duplicate rows while joining
geolocation =geolocation.groupby('geolocation_zip_code_prefix')[['geolocation_lat','geolocation_lng']].mean().reset_index()

# now duplicates for this column is removed using the groupby. Now this column can be used a primary key for geolocation
geolocation['geolocation_zip_code_prefix'].duplicated().sum()

np.int64(0)

#### Data type conversion

In [67]:
# data type conversion for date columns
# creating date columns for different tables
orders_date_cols=['order_purchase_timestamp','order_approved_at','order_delivered_carrier_date',
                   'order_delivered_customer_date','order_estimated_delivery_date']
order_reviews_date_cols=['review_creation_date','review_answer_timestamp']

def convert_to_datetime(df, cols):
    nulls_before = df[cols].isnull().sum()
    df[cols] = df[cols].apply(pd.to_datetime)
    nulls_after = df[cols].isnull().sum()
    return pd.DataFrame({'nulls_before': nulls_before, 'nulls_after': nulls_after, 'dtype':df[cols].dtypes})

date_check = pd.concat({
    'orders': convert_to_datetime(orders, orders_date_cols),
    'order_items': convert_to_datetime(order_items, ['shipping_limit_date']),
    'order_reviews': convert_to_datetime(order_reviews, order_reviews_date_cols),
})
date_check

# here we can see in the output that all the date columns have been converted to datetime format
# The nulls before and after are equal meaning all are converted properly

nulls_before  nulls_after  \
orders        order_purchase_timestamp                  0            0   
              order_approved_at                       160          160   
              order_delivered_carrier_date           1783         1783   
              order_delivered_customer_date          2965         2965   
              order_estimated_delivery_date             0            0   
order_items   shipping_limit_date                       0            0   
order_reviews review_creation_date                      0            0   
              review_answer_timestamp                   0            0   

                                                      dtype  
orders        order_purchase_timestamp       datetime64[ns]  
              order_approved_at              datetime64[ns]  
              order_delivered_carrier_date   datetime64[ns]  
              order_delivered_customer_date  datetime64[ns]  
              order_estimated_delivery_date  datetime64[ns]  
order_items   shipping_limit_date            datetime64[ns]  
order_reviews review_creation_date           datetime64[ns]  
              review_answer_timestamp        datetime64[ns]

In [68]:
# zip_code data type change from int to str as it'll remove zeros

customers['customer_zip_code_prefix']=customers['customer_zip_code_prefix'].astype(str).str.zfill(5)
sellers['seller_zip_code_prefix']=sellers['seller_zip_code_prefix'].astype(str).str.zfill(5)
geolocation['geolocation_zip_code_prefix']=geolocation['geolocation_zip_code_prefix'].astype(str).str.zfill(5)

# all the zip_codes will be converted to 5 digits or str type.
# for verification use this code
print(customers['customer_zip_code_prefix'].str.len().value_counts())
print(sellers['seller_zip_code_prefix'].str.len().value_counts())
print(geolocation['geolocation_zip_code_prefix'].str.len().value_counts())

customer_zip_code_prefix
5    99441
Name: count, dtype: int64
seller_zip_code_prefix
5    3095
Name: count, dtype: int64
geolocation_zip_code_prefix
5    19015
Name: count, dtype: int64


#### Missing values handling

In [69]:
# same code copied to see the nulls
# A function is created to perform these operations, instead of repeating same code for all the tables.
def column_info(df):
    # all these results are converted to a dataframe for better visibility
    info= pd.DataFrame({'dtype':df.dtypes, 'nulls':df.isnull().sum(), 'null_%':df.isnull().mean()*100, 'unique_count':df.nunique()})
    return info
# all the tables info are executed using the loop and combined using 'pd.concat' 
all_column_info= pd.concat({table_name:column_info(df) for table_name,df in dfs.items()})
# boolean indexing to check only the null columns info
all_column_info[all_column_info['nulls']>0]

dtype  nulls     null_%  \
order_reviews review_comment_title                   object  87656  88.341530   
              review_comment_message                 object  58247  58.702532   
orders        order_approved_at              datetime64[ns]    160   0.160899   
              order_delivered_carrier_date   datetime64[ns]   1783   1.793023   
              order_delivered_customer_date  datetime64[ns]   2965   2.981668   
products      product_category_name                  object    610   1.851234   
              product_name_lenght                   float64    610   1.851234   
              product_description_lenght            float64    610   1.851234   
              product_photos_qty                    float64    610   1.851234   
              product_weight_g                      float64      2   0.006070   
              product_length_cm                     float64      2   0.006070   
              product_height_cm                     float64      2   0.006070   
              product_width_cm                      float64      2   0.006070   

                                             unique_count  
order_reviews review_comment_title                   4527  
              review_comment_message                36159  
orders        order_approved_at                     90733  
              order_delivered_carrier_date          81018  
              order_delivered_customer_date         95664  
products      product_category_name                    73  
              product_name_lenght                      66  
              product_description_lenght             2960  
              product_photos_qty                       19  
              product_weight_g                       2204  
              product_length_cm                        99  
              product_height_cm                       102  
              product_width_cm                         95

1. Here, review_comment_title, review_comment_message will be left as NaN as it'll be easy during analysis in SQL.
2. All the null from date columns in the orders table will be left same as they might be meaningful nulls.
3. Nulls from product_name_length, product_description_length, product_photos_qty will be left NaN as any imputation changes the meaning. 

In [70]:
# Product_category_name null handling
products['product_category_name']=products['product_category_name'].fillna('unknown')

In [71]:
# convertinng the weights of 0 gms to nan so that it can be converted to median in next step
import numpy as np
products.loc[products['product_weight_g']<=0,'product_weight_g']=np.nan

# filling the remaining columns with median as those columns contain outliers
prod_nafill_cols=['product_weight_g','product_length_cm','product_height_cm','product_width_cm']
for col in prod_nafill_cols:
    products[col]=products[col].fillna(products[col].median())

#### Inconsistent values handling

In [72]:
#all inconsistent rules from previous section are listed and modifed here
#before converting
print((order_payments['payment_value']<=0).sum())
print((order_payments['payment_installments']<1).sum())
print((order_payments['payment_type']=='not_defined').sum())
print(order_payments.shape)

#inconsistent columns modification
order_payments.loc[order_payments['payment_installments']<=0, 'payment_installments']=1
order_payments=order_payments[order_payments['payment_value']>0]
order_payments=order_payments[order_payments['payment_type']!='not_defined']

#after converting
print((order_payments['payment_value']<=0).sum())
print((order_payments['payment_installments']<1).sum())
print((order_payments['payment_type']=='not_defined').sum())
print(order_payments.shape)

9
2
3
(103886, 5)
0
0
0
(103877, 5)


In [73]:
# these categories are missing from product_category_name of category_translation table
missing = ~products['product_category_name'].isin(category_translation['product_category_name'])
print(products.loc[missing, 'product_category_name'].value_counts())

product_category_name
unknown                                          610
portateis_cozinha_e_preparadores_de_alimentos     10
pc_gamer                                           3
Name: count, dtype: int64


In [74]:
# here we can see this category is missing
category_translation.loc[category_translation['product_category_name']=='portateis_cozinha_e_preparadores_de_alimentos']

,product_category_name,product_category_name_english


In [75]:
new_rows = pd.DataFrame({'product_category_name':['pc_gamer','portateis_cozinha_e_preparadores_de_alimentos','unknown'],
                         'product_category_name_english':['pc_gamer','portable_kitchen_and_food_preparers','unknown']})
category_translation = pd.concat([category_translation, new_rows], ignore_index=True)

In [ ]:
# after translation we can see this category
category_translation.loc[category_translation['product_category_name']=='portateis_cozinha_e_preparadores_de_alimentos']

,product_category_name,product_category_name_english
72,portateis_cozinha_e_preparadores_de_alimentos,portable_kitchen_and_food_preparers


#### Rename columns

In [77]:
# column names are renamed
products.rename(columns={'product_name_lenght':'product_name_length','product_description_lenght':'product_name_length'},inplace=True)

In [78]:
products.columns

Index(['product_id', 'product_category_name', 'product_name_length',
       'product_name_length', 'product_photos_qty', 'product_weight_g',
       'product_length_cm', 'product_height_cm', 'product_width_cm'],
      dtype='object')

In [81]:
# making a dictionary of all the files to do certain operations combinely
dfs={"customers":customers,
     "geolocation":geolocation,
     "order_items": order_items,
     "order_payments": order_payments,
     "order_reviews": order_reviews,
     "orders": orders,
     "products": products,
     "sellers": sellers,
     "category_translation": category_translation}

In [82]:
# same code copied to see after changes
# A function is created to perform these operations, instead of repeating same code for all the tables.
def column_info(df):
    # all these results are converted to a dataframe for better visibility
    info= pd.DataFrame({'dtype':df.dtypes, 'nulls':df.isnull().sum(), 'null_%':df.isnull().mean()*100, 'unique_count':df.nunique()})
    return info
# all the tables info are executed using the loop and combined using 'pd.concat' 
all_column_info= pd.concat({table_name:column_info(df) for table_name,df in dfs.items()})
all_column_info

dtype  nulls  \
customers            customer_id                            object      0   
                     customer_unique_id                     object      0   
                     customer_zip_code_prefix               object      0   
                     customer_city                          object      0   
                     customer_state                         object      0   
geolocation          geolocation_zip_code_prefix            object      0   
                     geolocation_lat                       float64      0   
                     geolocation_lng                       float64      0   
order_items          order_id                               object      0   
                     order_item_id                           int64      0   
                     product_id                             object      0   
                     seller_id                              object      0   
                     shipping_limit_date            datetime64[ns]      0   
                     price                                 float64      0   
                     freight_value                         float64      0   
order_payments       order_id                               object      0   
                     payment_sequential                      int64      0   
                     payment_type                           object      0   
                     payment_installments                    int64      0   
                     payment_value                         float64      0   
order_reviews        review_id                              object      0   
                     order_id                               object      0   
                     review_score                            int64      0   
                     review_comment_title                   object  87656   
                     review_comment_message                 object  58247   
                     review_creation_date           datetime64[ns]      0   
                     review_answer_timestamp        datetime64[ns]      0   
orders               order_id                               object      0   
                     customer_id                            object      0   
                     order_status                           object      0   
                     order_purchase_timestamp       datetime64[ns]      0   
                     order_approved_at              datetime64[ns]    160   
                     order_delivered_carrier_date   datetime64[ns]   1783   
                     order_delivered_customer_date  datetime64[ns]   2965   
                     order_estimated_delivery_date  datetime64[ns]      0   
products             product_id                             object      0   
                     product_category_name                  object      0   
                     product_name_length                   float64    610   
                     product_name_length                   float64    610   
                     product_photos_qty                    float64    610   
                     product_weight_g                      float64      0   
                     product_length_cm                     float64      0   
                     product_height_cm                     float64      0   
                     product_width_cm                      float64      0   
sellers              seller_id                              object      0   
                     seller_zip_code_prefix                 object      0   
                     seller_city                            object      0   
                     seller_state                           object      0   
category_translation product_category_name                  object      0   
                     product_category_name_english          object      0   

                                                       null_%  unique_count  
customers            customer_id                     0.

In [86]:
# loading the cleaned data to cleaned folder
customers.to_csv('../data/cleaned/customers.csv',index=False)
geolocation.to_csv('../data/cleaned/geolocation.csv',index=False)
order_items.to_csv('../data/cleaned/order_items.csv',index=False)
order_payments.to_csv('../data/cleaned/order_payments.csv',index=False)
order_reviews.to_csv('../data/cleaned/order_reviews.csv',index=False)
orders.to_csv('../data/cleaned/orders.csv',index=False)
products.to_csv('../data/cleaned/products.csv',index=False)
sellers.to_csv('../data/cleaned/sellers.csv',index=False)
category_translation.to_csv('../data/cleaned/product_category_name_translation.csv',index=False)